In [1]:
import math
import numpy as np
from PIL import Image
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.visualization import plot_histogram
from qiskit.circuit.library import MCXGate
from qiskit.circuit.library import RYGate
from qiskit import transpile
from qiskit_aer import Aer
import matplotlib.pyplot as plt
import os
import random

def load_gray_image_as_array(img_path, n):
    img = Image.open(img_path).convert('L') 
    img1 = img.resize((2**n, 2**n))
    img_array = np.array(img1)
    return img_array

def get_bit(value, bit_index):
    bit_str = format(value, '08b')
    bit_value = int(bit_str[7 - bit_index]) 
    return bit_value

img_path1 = "C:\\Users\\pc\\Desktop\\(2,2)QMVC-PBS1\\1 secret image.png"  
img_path2 = "C:\\Users\\pc\\Desktop\\(2,2)QMVC-PBS1\\1 cover image.png"  
n = 1 


# 调控因子
theta0  = 7 * math.pi / 16
theta00 = math.pi / 2
theta1  = 9 * math.pi / 16


img1 = load_gray_image_as_array(img_path1, n)
img2 = load_gray_image_as_array(img_path2, n)


backend = Aer.get_backend('qasm_simulator')

# SC1
color_qubits      = QuantumRegister(1, name="c")
bitplane_qubits   = QuantumRegister(3, name="l")
x_qubits          = QuantumRegister(n, name='x')
y_qubits          = QuantumRegister(n, name='y')

# SC2
color_qubits1     = QuantumRegister(1, name="c'")
bitplane_qubits1  = QuantumRegister(3, name="l'")
x_qubits1         = QuantumRegister(n, name="x'")
y_qubits1         = QuantumRegister(n, name="y'")


qc = QuantumCircuit(
    color_qubits, bitplane_qubits, x_qubits,y_qubits,  
    color_qubits1, bitplane_qubits1, x_qubits1, y_qubits1
)
    

# SC1

for i in range(3): 
    qc.h(bitplane_qubits[i])
for i in range(n): 
    qc.h(x_qubits[i])
for i in range(n): 
    qc.h(y_qubits[i])

qc.barrier()


for l in range(8):
    for y in range(2 ** n):
        for x in range(2 ** n):
            
            s_bit = get_bit(img1[y, x], l)
            c_bit = get_bit(img2[y, x], l)
            
            l_bin = format(l, '03b')
            y_bin = format(y, f'0{n}b')
            x_bin = format(x, f'0{n}b')

            ctrl_qubits = list(y_qubits) + list(x_qubits) + list(bitplane_qubits)
            ctrl_state = l_bin + x_bin + y_bin
            

            if c_bit == 0:
                ry_gate0 = RYGate(theta0).control(len(ctrl_qubits), ctrl_state=ctrl_state)
                qc.append(ry_gate0, ctrl_qubits + list(color_qubits))
                
            elif c_bit == 1:
                ry_gate0 = RYGate(theta1).control(len(ctrl_qubits), ctrl_state=ctrl_state)
                qc.append(ry_gate0, ctrl_qubits + list(color_qubits))
                
    qc.barrier()

# SC2

for i in range(3):
    qc.cx(bitplane_qubits[i], bitplane_qubits1[i])

qc.cx(x_qubits, x_qubits1)
qc.cx(y_qubits, y_qubits1)


qc.barrier()


for l in range(8):
    for y in range(2 ** n):
        for x in range(2 ** n):

            s_bit = get_bit(img1[y, x], l)
            c_bit = get_bit(img2[y, x], l)

            l_bin = format(l, '03b')
            y_bin = format(y, f'0{n}b')
            x_bin = format(x, f'0{n}b')

            ctrl_qubits1 = list(y_qubits1) + list(x_qubits1) + list(bitplane_qubits1)
            ctrl_state1 = l_bin + x_bin + y_bin

            if c_bit == 0 and s_bit == 0:
                
                ctrl_qubits_mcx = ctrl_qubits1 + list(color_qubits)
                ctrl_state_mcx = '1' + ctrl_state1
                gate = MCXGate(len(ctrl_qubits_mcx), ctrl_state=ctrl_state_mcx)
                qc.append(gate, ctrl_qubits_mcx + list(color_qubits1))

            elif c_bit == 0 and s_bit == 1:

                ry_gate1 = RYGate(theta00).control(len(ctrl_qubits1), ctrl_state=ctrl_state1)
                qc.append(ry_gate1, ctrl_qubits1 + list(color_qubits1))

            elif c_bit == 1 and s_bit == 0:

                ctrl_qubits_mcx = ctrl_qubits1 + list(color_qubits)
                ctrl_state_mcx = '1' + ctrl_state1
                gate = MCXGate(len(ctrl_qubits_mcx), ctrl_state=ctrl_state_mcx)
                qc.append(gate, ctrl_qubits_mcx + list(color_qubits1))

            elif c_bit == 1 and s_bit == 1:

                ry_gate1 = RYGate(theta00).control(len(ctrl_qubits1), ctrl_state=ctrl_state1)
                qc.append(ry_gate1, ctrl_qubits1 + list(color_qubits1))

    qc.barrier()



qc.draw('mpl', cregbundle=False, fold=18, scale=1)